<a href="https://colab.research.google.com/github/sethkipsangmutuba/Data-Visualization/blob/main/Session_B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Seth Kipsang
-----

## Hierarchical Indexing

- A Multiply Indexed Series  
- Methods of MultiIndex Creation  
- Indexing and Slicing a MultiIndex  
- Rearranging Multi-Indices  
- Data Aggregations on Multi-Indices  
------

## Combining Datasets: Concat and Append

- Recall: Concatenation of NumPy Arrays  
- Simple Concatenation with `pd.concat`  

---

## Combining Datasets: Merge and Join

- Relational Algebra  
- Categories of Joins  
- Specification of the Merge Key  
- Specifying Set Arithmetic for Joins  
- Overlapping Column Names: The `suffixes` Keyword  


##1 Hierarchical Indexing

Enables representation of higher-dimensional data within Series and DataFrame.

Uses multiple index levels (MultiIndex) instead of separate dimensions.

More practical than higher-dimensional structures like Panel/Panel4D.

Allows compact storage and flexible data organization.

Supports advanced indexing, slicing, and aggregation.

Includes tools to convert between flat and hierarchical data formats.

##1.1 A Multiply Indexed Series (Shortened Notes)

We can represent two-dimensional data inside a one-dimensional Series using composite keys.

---

###1.1.1 The bad way

A simple approach is using tuples as index keys (state, year). This works for basic slicing and selection, but becomes inefficient and messy when filtering specific conditions (e.g., extracting all entries for a given year requires manual iteration over the index). It does not scale well for large datasets.

---

###1.1.2 The better way: MultiIndex

Pandas provides a structured solution called MultiIndex, which supports multiple index levels (e.g., state and year). After converting to a MultiIndex, data becomes easier to slice using partial indexing, such as selecting all values for a specific year directly and efficiently.

---

###1.1.3 MultiIndex as extra dimension

MultiIndex allows a Series to behave like higher-dimensional data. The same data can be reshaped into a DataFrame and back again, showing that hierarchical indexing is equivalent to adding extra dimensions.

It also makes it easy to extend datasets (e.g., adding new variables like under-18 population) while keeping structure intact. Standard operations such as arithmetic between columns work naturally, enabling fast and flexible analysis of multi-dimensional data.

LOAD IRIS DATASET

In [512]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

df["target"] = iris.target

THE "BAD WAY": TUPLE-BASED INDEX

In [513]:
# create composite key: (species, sample_id)

tuple_index = [(f"species_{df.loc[i, 'target']}", i) for i in df.index]

series_bad = pd.Series(df["sepal length (cm)"].values, index=tuple_index)

print("\n--- BAD WAY (Tuple Index) ---")
print(series_bad.head())

# extracting all values for species_0 (inefficient)
filtered_bad = [v for (k, v) in zip(series_bad.index, series_bad.values) if k[0] == "species_0"]

print("\nManual filtering (inefficient):")
print(filtered_bad[:5])


--- BAD WAY (Tuple Index) ---
(species_0, 0)    5.1
(species_0, 1)    4.9
(species_0, 2)    4.7
(species_0, 3)    4.6
(species_0, 4)    5.0
dtype: float64

Manual filtering (inefficient):
[np.float64(5.1), np.float64(4.9), np.float64(4.7), np.float64(4.6), np.float64(5.0)]


THE BETTER WAY: MultiIndex

In [514]:
multi_index = pd.MultiIndex.from_arrays(
    [df["target"], df.index],
    names=["species", "sample_id"]
)

series_multi = pd.Series(df["sepal length (cm)"].values, index=multi_index)
series_multi

species  sample_id
0        0            5.1
         1            4.9
         2            4.7
         3            4.6
         4            5.0
                     ... 
2        145          6.7
         146          6.3
         147          6.5
         148          6.2
         149          5.9
Length: 150, dtype: float64

MULTIINDEX AS EXTRA DIMENSION

In [515]:
# convert to DataFrame (unstack)
df_multi = series_multi.unstack(level=0)
df_multi

species,0,1,2
sample_id,,,
0,5.1,NaN,NaN
1,4.9,NaN,NaN
2,4.7,NaN,NaN
3,4.6,NaN,NaN
4,5.0,NaN,NaN
...,...,...,...
145,NaN,NaN,6.7
146,NaN,NaN,6.3
147,NaN,NaN,6.5


In [516]:
# stack back to Series
series_restored = df_multi.stack()
series_restored

,,0
sample_id,species,
0,0,5.1
1,0,4.9
2,0,4.7
3,0,4.6
4,0,5.0
...,...,...
145,2,6.7
146,2,6.3
147,2,6.5


EXTENDING DATA (ADDING NEW VARIABLE)

In [517]:
# create another variable (e.g., petal length)
series_petal = pd.Series(
    df["petal length (cm)"].values,
    index=multi_index
)

# combine into DataFrame
combined = pd.DataFrame({
    "sepal_length": series_multi,
    "petal_length": series_petal
})

# arithmetic operation
combined["ratio"] = combined["sepal_length"] / combined["petal_length"]

print("\n---- EXTENDED MULTIINDEX DATA -----")
print(combined.head())


---- EXTENDED MULTIINDEX DATA -----
                   sepal_length  petal_length     ratio
species sample_id                                      
0       0                   5.1           1.4  3.642857
        1                   4.9           1.4  3.500000
        2                   4.7           1.3  3.615385
        3                   4.6           1.5  3.066667
        4                   5.0           1.4  3.571429


## Key Insight

- Tuple indexing works but is inefficient and not scalable  
- MultiIndex enables structured hierarchical indexing  
- Supports fast partial selection (e.g., by species)  
- Behaves like higher-dimensional data  
- Easily reshaped between Series and DataFrame  
- Enables clean extension and vectorized operations  

##1.2 Methods of MultiIndex Creation

The most straightforward way to build a MultiIndex Series or DataFrame is to pass multiple index arrays directly to the constructor. Pandas automatically combines them into hierarchical levels in the background.

Another common approach is using a dictionary with tuple keys. Pandas detects tuple structures and converts them into a MultiIndex automatically, producing a hierarchical structure without manual specification.

---

##1.2.1 Explicit MultiIndex Constructors

For more control, Pandas provides explicit constructors:

- From arrays: each array represents one level of the index  
- From tuples: each tuple defines a full coordinate across levels  
- From Cartesian product: generates all possible combinations of index levels automatically  
- From internal encoding: using predefined levels and label mappings for fine-grained control  

All these methods can be used when creating or reindexing Series and DataFrames.

---

##1.2.2 MultiIndex Level Names

Index levels can be named to improve clarity and interpretability. Names can be assigned during construction or set afterward. This is especially useful in complex datasets to clearly indicate what each level represents.

---

##1.2.3 MultiIndex for Columns

MultiIndexing also applies to DataFrame columns, not just rows. Rows and columns are symmetric in Pandas, so both can carry hierarchical structure.

In a multi-level column setup, data can represent higher-dimensional structures (e.g., subject, measurement type, year, visit). This allows compact representation of complex datasets, such as medical or longitudinal studies.

You can easily select data for a specific top-level column (e.g., one subject), retrieving all associated sub-measurements across the dataset.

 LOAD IRIS DATASET

In [518]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

df["target"] = iris.target

DIRECT CONSTRUCTION (MULTIPLE ARRAYS → AUTOMATIC MULTIINDEX)

In [519]:
arrays = [df["target"], df.index]

series_direct = pd.Series(
    df["sepal length (cm)"].values,
    index=arrays
)
series_direct

target     
0       0      5.1
        1      4.9
        2      4.7
        3      4.6
        4      5.0
              ... 
2       145    6.7
        146    6.3
        147    6.5
        148    6.2
        149    5.9
Length: 150, dtype: float64

FROM DICTIONARY WITH TUPLE KEYS (AUTO DETECTION)

In [520]:
data_dict = {
    (df.loc[i, "target"], i): df.loc[i, "sepal width (cm)"]
    for i in range(10)
}

series_dict = pd.Series(data_dict)
series_dict

0  0    3.5
   1    3.0
   2    3.2
   3    3.1
   4    3.6
   5    3.9
   6    3.4
   7    3.4
   8    2.9
   9    3.1
dtype: float64

EXPLICIT MULTIINDEX CONSTRUCTORS

In [521]:
# --- from_arrays ---
mi_arrays = pd.MultiIndex.from_arrays(
    [df["target"], df.index],
    names=["species", "sample_id"]
)

series_arrays = pd.Series(df["petal length (cm)"].values, index=mi_arrays)
series_arrays

species  sample_id
0        0            1.4
         1            1.4
         2            1.3
         3            1.5
         4            1.4
                     ... 
2        145          5.2
         146          5.0
         147          5.2
         148          5.4
         149          5.1
Length: 150, dtype: float64

In [522]:
# --- from_tuples ---
tuples = [(df.loc[i, "target"], i) for i in df.index]
mi_tuples = pd.MultiIndex.from_tuples(tuples, names=["species", "sample_id"])

series_tuples = pd.Series(df["petal width (cm)"].values, index=mi_tuples)
series_tuples

species  sample_id
0        0            0.2
         1            0.2
         2            0.2
         3            0.2
         4            0.2
                     ... 
2        145          2.3
         146          1.9
         147          2.0
         148          2.3
         149          1.8
Length: 150, dtype: float64

In [523]:
# --- from_product (Cartesian product) ---
mi_product = pd.MultiIndex.from_product(
    [[0, 1, 2], ["A", "B"]],
    names=["species", "group"]
)

series_product = pd.Series(np.random.rand(len(mi_product)), index=mi_product)
series_product

species  group
0        A        0.745434
         B        0.141070
1        A        0.511412
         B        0.469680
2        A        0.786714
         B        0.650791
dtype: float64

MULTIINDEX LEVEL NAMES

In [524]:
series_arrays.index.names

FrozenList(['species', 'sample_id'])

MULTIINDEX FOR COLUMNS

In [525]:
# create hierarchical columns
columns = pd.MultiIndex.from_product(
    [["sepal", "petal"], ["length", "width"]],
    names=["feature", "measurement"]
)

df_multi_col = pd.DataFrame(df.iloc[:, :4].values, columns=columns)
df_multi_col

feature      sepal        petal      
measurement length width length width
0              5.1   3.5    1.4   0.2
1              4.9   3.0    1.4   0.2
2              4.7   3.2    1.3   0.2
3              4.6   3.1    1.5   0.2
4              5.0   3.6    1.4   0.2
..             ...   ...    ...   ...
145            6.7   3.0    5.2   2.3
146            6.3   2.5    5.0   1.9
147            6.5   3.0    5.2   2.0
148            6.2   3.4    5.4   2.3
149            5.9   3.0    5.1   1.8

[150 rows x 4 columns]

In [526]:
# selecting all "sepal" measurements
print("\nSelect top-level column (sepal):")
print(df_multi_col["sepal"].head())


Select top-level column (sepal):
measurement  length  width
0               5.1    3.5
1               4.9    3.0
2               4.7    3.2
3               4.6    3.1
4               5.0    3.6


## Key Insight

- MultiIndex can be created automatically or explicitly  
- `from_arrays`, `from_tuples`, `from_product` give structured control  
- Dictionary with tuple keys auto-generates MultiIndex  
- Level names improve clarity in hierarchical data  
- MultiIndex applies to both rows and columns  
- Enables compact representation of higher-dimensional datasets  

##1.3 Indexing and Slicing a MultiIndex

Indexing and slicing a MultiIndex is intuitive if you view it as extra dimensions. It applies to both Series and DataFrames.

---

##1.3.1 Multiply Indexed Series

Consider a Series of state populations indexed by state and year:

You can access a single value using multiple keys:

- A specific state and year returns one value.

Partial indexing is also possible:

- Selecting only the state returns a smaller Series grouped by year.  
- Selecting only the year (with proper ordering) returns all states for that year.

Slicing works when the index is sorted:

- You can slice across states or across ranges of index levels.

You can also:

- Filter using Boolean conditions (e.g., values above a threshold)  
- Use fancy indexing to select multiple states at once  

---

##1.3.2 Multiply Indexed DataFrames

In DataFrames, columns behave as a primary indexing level.

You can:

- Select a specific subject and measurement (e.g., a person’s HR data)  
- Extract a single column from a MultiIndex structure easily  

Standard indexers (like positional or label-based selection) still work:

- You can select rows and columns using standard slicing  
- You can pass tuples to select specific column-level combinations  

However:

- Complex slicing inside tuples is not directly supported in a clean syntax  

To handle this, Pandas provides:

- A specialized `IndexSlice` tool for advanced multi-level slicing  

Example use:

- Selecting all rows for a given visit  
- Selecting only heart rate data across all subjects  

---

## Insight

MultiIndex slicing treats hierarchical data as layered dimensions, enabling compact and flexible selection across complex datasets.

In [527]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

df["target"] = iris.target

CREATE MULTIINDEX SERIES (species, sample_id)

In [528]:
mi = pd.MultiIndex.from_arrays(
    [df["target"], df.index],
    names=["species", "sample_id"]
)

series = pd.Series(df["sepal length (cm)"].values, index=mi)

# IMPORTANT: sort index for slicing
series = series.sort_index()
series

species  sample_id
0        0            5.1
         1            4.9
         2            4.7
         3            4.6
         4            5.0
                     ... 
2        145          6.7
         146          6.3
         147          6.5
         148          6.2
         149          5.9
Length: 150, dtype: float64

 INDEXING (SINGLE + PARTIAL)

In [529]:
# single value
print("\nSingle value (species=0, sample=5):")
print(series.loc[(0, 5)])


Single value (species=0, sample=5):
5.4


In [530]:
# partial indexing → all samples for species 0
print("\nAll samples for species 0:")
print(series.loc[0].head())


All samples for species 0:
sample_id
0    5.1
1    4.9
2    4.7
3    4.6
4    5.0
dtype: float64


In [531]:
# selecting all species for a given sample_id
print("\nAll species for sample_id=10:")
print(series.xs(10, level="sample_id"))


All species for sample_id=10:
species
0    5.4
dtype: float64


SLICING (REQUIRES SORTED INDEX)

In [532]:
# slice across species
print("\nSlice species 0 to 1:")
print(series.loc[0:1].head())


Slice species 0 to 1:
species  sample_id
0        0            5.1
         1            4.9
         2            4.7
         3            4.6
         4            5.0
dtype: float64


In [533]:
# slice across both levels
print("\nSlice (species 0–1, sample 0–10):")
print(series.loc[(slice(0,1), slice(0,10))])


Slice (species 0–1, sample 0–10):
species  sample_id
0        0            5.1
         1            4.9
         2            4.7
         3            4.6
         4            5.0
         5            5.4
         6            4.6
         7            5.0
         8            4.4
         9            4.9
         10           5.4
dtype: float64


BOOLEAN FILTERING

In [534]:
filtered = series[series > 6.5]
filtered

species  sample_id
1        50           7.0
         52           6.9
         58           6.6
         65           6.7
         75           6.6
         76           6.8
         77           6.7
         86           6.7
2        102          7.1
         105          7.6
         107          7.3
         108          6.7
         109          7.2
         112          6.8
         117          7.7
         118          7.7
         120          6.9
         122          7.7
         124          6.7
         125          7.2
         129          7.2
         130          7.4
         131          7.9
         135          7.7
         139          6.9
         140          6.7
         141          6.9
         143          6.8
         144          6.7
         145          6.7
dtype: float64

FANCY INDEXING

In [535]:
# select multiple species
fancy = series.loc[[0, 2]]

print("\nFancy indexing (species 0 and 2):")
print(fancy.head())


Fancy indexing (species 0 and 2):
species  sample_id
0        0            5.1
         1            4.9
         2            4.7
         3            4.6
         4            5.0
dtype: float64


MULTIINDEX DATAFRAME (COLUMNS)

In [536]:
columns = pd.MultiIndex.from_product(
    [["sepal", "petal"], ["length", "width"]],
    names=["feature", "measurement"]
)

df_multi = pd.DataFrame(df.iloc[:, :4].values, columns=columns)
df_multi

feature      sepal        petal      
measurement length width length width
0              5.1   3.5    1.4   0.2
1              4.9   3.0    1.4   0.2
2              4.7   3.2    1.3   0.2
3              4.6   3.1    1.5   0.2
4              5.0   3.6    1.4   0.2
..             ...   ...    ...   ...
145            6.7   3.0    5.2   2.3
146            6.3   2.5    5.0   1.9
147            6.5   3.0    5.2   2.0
148            6.2   3.4    5.4   2.3
149            5.9   3.0    5.1   1.8

[150 rows x 4 columns]

COLUMN INDEXING

In [537]:
# select all sepal data
print("\nSelect all 'sepal':")
print(df_multi["sepal"].head())


Select all 'sepal':
measurement  length  width
0               5.1    3.5
1               4.9    3.0
2               4.7    3.2
3               4.6    3.1
4               5.0    3.6


In [538]:
# select specific column (tuple)
print("\nSelect ('petal', 'length'):")
print(df_multi[("petal", "length")].head())


Select ('petal', 'length'):
0    1.4
1    1.4
2    1.3
3    1.5
4    1.4
Name: (petal, length), dtype: float64


ADVANCED SLICING WITH IndexSlice

In [539]:
idx = pd.IndexSlice

# select all rows, only petal measurements
print("\nIndexSlice: select all petal measurements")
print(df_multi.loc[:, idx["petal", :]].head())


IndexSlice: select all petal measurements
feature      petal      
measurement length width
0              1.4   0.2
1              1.4   0.2
2              1.3   0.2
3              1.5   0.2
4              1.4   0.2


## Key Insight

- MultiIndex behaves like extra dimensions  
- Supports single, partial, and hierarchical indexing  
- Slicing requires sorted index  
- Boolean and fancy indexing work naturally  
- DataFrame columns can also be hierarchical  
- IndexSlice enables clean advanced multi-level slicing  

##1.4 Rearranging Multi-Indices

One of the key ideas in working with hierarchical data is transforming its structure without losing information. These operations mainly reshape how data is viewed for analysis and computation. Earlier we briefly saw `stack()` and `unstack()`, but Pandas offers more control for switching between multi-index forms and standard tabular layouts.

---

###1.4.1 Sorted and unsorted indices

A critical detail is that many MultiIndex operations only work correctly when the index is sorted. If the ordering is random or unsorted, slicing and partial selection can fail.

For example, if a MultiIndex is created without ordering, attempts to slice across index ranges may raise errors because Pandas requires lexicographical structure for safe navigation across levels.

Once the index is sorted, operations such as partial slicing begin to behave predictably and efficiently. Sorting ensures that hierarchical levels are organized consistently, which is essential for performance and correct indexing behavior.

---

##1.4.2 Stacking and unstacking indices

The `stack()` and `unstack()` methods allow movement between “long” (hierarchical Series) and “wide” (DataFrame) formats. Unstacking shifts one level of the index into columns, effectively converting part of the hierarchy into a tabular structure. Stacking reverses this process, restoring the original multi-level index representation.

This flexibility is useful when switching between analytical perspectives—for example, comparing values across years versus comparing across states in a matrix-like layout.

---

###1.4.3 Index setting and resetting

Another important transformation is moving between index levels and columns. `reset_index()` converts index levels back into regular columns, producing a flat DataFrame that is often closer to raw data formats encountered in practice.

Conversely, `set_index()` promotes one or more columns into a hierarchical index, reconstructing structured relationships within the dataset. This is especially common in real-world workflows where data arrives in flat tables but requires hierarchical organization for analysis.

---

## Practical perspective

In practice, these two operations; resetting and setting indices are among the most frequently used tools for preparing messy datasets into structured analytical forms suitable for Pandas workflows.

LOAD IRIS DATASET

In [540]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

df["target"] = iris.target

CREATE MULTIINDEX (species, sample_id)

In [541]:
df_mi = df.set_index(["target", df.index])
df_mi.index.names = ["species", "sample_id"]
df_mi

sepal length (cm)  sepal width (cm)  petal length (cm)  \
species sample_id                                                           
0       0                        5.1               3.5                1.4   
        1                        4.9               3.0                1.4   
        2                        4.7               3.2                1.3   
        3                        4.6               3.1                1.5   
        4                        5.0               3.6                1.4   
...                              ...               ...                ...   
2       145                      6.7               3.0                5.2   
        146                      6.3               2.5                5.0   
        147                      6.5               3.0                5.2   
        148                      6.2               3.4                5.4   
        149                      5.9               3.0                5.1   

                   petal width (cm)  
species sample_id                    
0       0                       0.2  
        1                       0.2  
        2                       0.2  
        3                       0.2  
        4                       0.2  
...                             ...  
2       145                     2.3  
        146                     1.9  
        147                     2.0  
        148                     2.3  
        149                     1.8  

[150 rows x 4 columns]

UNSORTED vs SORTED INDEX

In [542]:
# shuffle (unsorted)
df_unsorted = df_mi.sample(frac=1)
df_unsorted

,,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
species,sample_id,,,,
0,14,5.8,4.0,1.2,0.2
1,80,5.5,2.4,3.8,1.1
2,143,6.8,3.2,5.9,2.3
1,50,7.0,3.2,4.7,1.4
2,146,6.3,2.5,5.0,1.9
...,...,...,...,...,...
1,54,6.5,2.8,4.6,1.5
2,149,5.9,3.0,5.1,1.8
1,90,5.5,2.6,4.4,1.2


In [543]:
# slicing fails on unsorted index
try:
    print("\nAttempt slicing unsorted index:")
    print(df_unsorted.loc[(0, slice(0,10)), :])
except Exception as e:
    print("Error due to unsorted index:", e)

# sort index
df_sorted = df_unsorted.sort_index()
df_sorted


Attempt slicing unsorted index:
Error due to unsorted index: 'MultiIndex slicing requires the index to be lexsorted: slicing on levels [1], lexsort depth 0'


sepal length (cm)  sepal width (cm)  petal length (cm)  \
species sample_id                                                           
0       0                        5.1               3.5                1.4   
        1                        4.9               3.0                1.4   
        2                        4.7               3.2                1.3   
        3                        4.6               3.1                1.5   
        4                        5.0               3.6                1.4   
...                              ...               ...                ...   
2       145                      6.7               3.0                5.2   
        146                      6.3               2.5                5.0   
        147                      6.5               3.0                5.2   
        148                      6.2               3.4                5.4   
        149                      5.9               3.0                5.1   

                   petal width (cm)  
species sample_id                    
0       0                       0.2  
        1                       0.2  
        2                       0.2  
        3                       0.2  
        4                       0.2  
...                             ...  
2       145                     2.3  
        146                     1.9  
        147                     2.0  
        148                     2.3  
        149                     1.8  

[150 rows x 4 columns]

 CORRECT MULTIINDEX SLICING

In [544]:
idx = pd.IndexSlice

print("\nSlicing after sorting (correct):")
df_sorted.loc[idx[0, 0:10], :]


Slicing after sorting (correct):


sepal length (cm)  sepal width (cm)  petal length (cm)  \
species sample_id                                                           
0       0                        5.1               3.5                1.4   
        1                        4.9               3.0                1.4   
        2                        4.7               3.2                1.3   
        3                        4.6               3.1                1.5   
        4                        5.0               3.6                1.4   
        5                        5.4               3.9                1.7   
        6                        4.6               3.4                1.4   
        7                        5.0               3.4                1.5   
        8                        4.4               2.9                1.4   
        9                        4.9               3.1                1.5   
        10                       5.4               3.7                1.5   

                   petal width (cm)  
species sample_id                    
0       0                       0.2  
        1                       0.2  
        2                       0.2  
        3                       0.2  
        4                       0.2  
        5                       0.4  
        6                       0.3  
        7                       0.2  
        8                       0.2  
        9                       0.1  
        10                      0.2

STACKING AND UNSTACKING

In [545]:
# unstack → species to columns
df_unstacked = df_sorted.unstack(level=0)
df_unstacked

sepal length (cm)          sepal width (cm)           \
species                   0   1    2                0   1    2   
sample_id                                                        
0                       5.1 NaN  NaN              3.5 NaN  NaN   
1                       4.9 NaN  NaN              3.0 NaN  NaN   
2                       4.7 NaN  NaN              3.2 NaN  NaN   
3                       4.6 NaN  NaN              3.1 NaN  NaN   
4                       5.0 NaN  NaN              3.6 NaN  NaN   
...                     ...  ..  ...              ...  ..  ...   
145                     NaN NaN  6.7              NaN NaN  3.0   
146                     NaN NaN  6.3              NaN NaN  2.5   
147                     NaN NaN  6.5              NaN NaN  3.0   
148                     NaN NaN  6.2              NaN NaN  3.4   
149                     NaN NaN  5.9              NaN NaN  3.0   

          petal length (cm)          petal width (cm)           
species                   0   1    2                0   1    2  
sample_id                                                       
0                       1.4 NaN  NaN              0.2 NaN  NaN  
1                       1.4 NaN  NaN              0.2 NaN  NaN  
2                       1.3 NaN  NaN              0.2 NaN  NaN  
3                       1.5 NaN  NaN              0.2 NaN  NaN  
4                       1.4 NaN  NaN              0.2 NaN  NaN  
...                     ...  ..  ...              ...  ..  ...  
145                     NaN NaN  5.2              NaN NaN  2.3  
146                     NaN NaN  5.0              NaN NaN  1.9  
147                     NaN NaN  5.2              NaN NaN  2.0  
148                     NaN NaN  5.4              NaN NaN  2.3  
149                     NaN NaN  5.1              NaN NaN  1.8  

[150 rows x 12 columns]

In [546]:
# stack back
df_stacked = df_unstacked.stack()
df_stacked

,,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
sample_id,species,,,,
0,0,5.1,3.5,1.4,0.2
1,0,4.9,3.0,1.4,0.2
2,0,4.7,3.2,1.3,0.2
3,0,4.6,3.1,1.5,0.2
4,0,5.0,3.6,1.4,0.2
...,...,...,...,...,...
145,2,6.7,3.0,5.2,2.3
146,2,6.3,2.5,5.0,1.9
147,2,6.5,3.0,5.2,2.0


RESETTING AND SETTING INDEX

In [547]:
# reset index
df_reset = df_sorted.reset_index()
df_reset

,species,sample_id,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,0,0,5.1,3.5,1.4,0.2
1,0,1,4.9,3.0,1.4,0.2
2,0,2,4.7,3.2,1.3,0.2
3,0,3,4.6,3.1,1.5,0.2
4,0,4,5.0,3.6,1.4,0.2
...,...,...,...,...,...,...
145,2,145,6.7,3.0,5.2,2.3
146,2,146,6.3,2.5,5.0,1.9
147,2,147,6.5,3.0,5.2,2.0
148,2,148,6.2,3.4,5.4,2.3


In [548]:
# reconstruct MultiIndex
df_reconstructed = df_reset.set_index(["species", "sample_id"])
df_reconstructed

sepal length (cm)  sepal width (cm)  petal length (cm)  \
species sample_id                                                           
0       0                        5.1               3.5                1.4   
        1                        4.9               3.0                1.4   
        2                        4.7               3.2                1.3   
        3                        4.6               3.1                1.5   
        4                        5.0               3.6                1.4   
...                              ...               ...                ...   
2       145                      6.7               3.0                5.2   
        146                      6.3               2.5                5.0   
        147                      6.5               3.0                5.2   
        148                      6.2               3.4                5.4   
        149                      5.9               3.0                5.1   

                   petal width (cm)  
species sample_id                    
0       0                       0.2  
        1                       0.2  
        2                       0.2  
        3                       0.2  
        4                       0.2  
...                             ...  
2       145                     2.3  
        146                     1.9  
        147                     2.0  
        148                     2.3  
        149                     1.8  

[150 rows x 4 columns]

## Key Insight

- MultiIndex must be sorted for slicing  
- Use `pd.IndexSlice` for correct hierarchical slicing  
- `unstack()` and `stack()` reshape between long and wide formats  
- `reset_index()` flattens structure  
- `set_index()` rebuilds hierarchy  
- Enables flexible restructuring of complex datasets  

## Panel Data

Pandas includes `pd.Panel` and `pd.Panel4D` for 3D and 4D data structures, extending Series and DataFrame concepts. They support similar indexing tools like `loc` and `iloc`.

However, they are rarely used in practice today. Hierarchical indexing (MultiIndex) is preferred because it is simpler, more flexible, and integrates better with Series and DataFrames.

Panels also rely on dense storage, which can be inefficient for real-world datasets. MultiIndexing instead provides a more efficient, sparse way to represent higher-dimensional data.

##2. Combining Datasets: Concat and Append

Combining data from multiple sources is a core operation in data analysis. This can range from simple stacking of datasets to more complex database-style merging where overlaps must be handled carefully. Pandas is designed to support these operations efficiently.

The main tool for simple combination is `pd.concat`, which joins Series or DataFrames along a particular axis. This is commonly used when datasets share a similar structure.

For illustration, a small helper function is often used to quickly generate structured DataFrames for testing and demonstration purposes.

##2.1 Recall: Concatenation of NumPy Arrays

Concatenation of Series and DataFrame objects is similar to NumPy array concatenation using `np.concatenate`. It combines multiple arrays into one:

The main input is a list/tuple of arrays, and you can control the axis along which concatenation occurs.

For example, 1D arrays can be merged into a single longer array, while 2D arrays can be joined either row-wise or column-wise depending on the axis setting.

This same idea extends directly to Pandas structures, forming the basis for dataset combination operations.

##2.2 Simple Concatenation with pd.concat (short notes)

`pd.concat()` is used to combine Series/DataFrames, similar to NumPy concatenation but more flexible.

Works for:
- Series (vertical stacking)
- DataFrames (row-wise or column-wise combination)

Default behavior:
- Concatenates along rows (`axis=0`)

Axis control:
- Row-wise vs column-wise concatenation

Key behaviors:
- Preserves original indices (can create duplicates)
- Missing values appear as NaN when columns don’t match

Handling duplicate indices:
- `verify_integrity=True` → raises error if duplicates exist  
- `ignore_index=True` → resets index to simple integers  
- `keys=[...]` → creates hierarchical (MultiIndex) labels  

Column alignment options:
- `join='outer'` (default): union of columns  
- `join='inner'`: intersection of columns only  

Column restriction option:
- `join_axes` (older approach): forces specific columns  

Append shortcut:
- `df.append()` = simplified concat (row-wise only)  
- Does NOT modify original object  
- Less efficient than `pd.concat()` for repeated use

Load Dataset

In [549]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)
df["target"] = iris.target

CREATE MULTIPLE DATA SOURCES (IRIS SPLITS)

In [550]:
df_a = df.iloc[:50]
df_b = df.iloc[50:100]
df_c = df.iloc[100:150]

print("\n--- IRIS SPLITS ---")
print(df_a.shape, df_b.shape, df_c.shape)


--- IRIS SPLITS ---
(50, 5) (50, 5) (50, 5)


NUMPY CONCATENATION RECALL

In [551]:
x = np.array([1, 2, 3])
y = np.array([4, 5, 6])
np.concatenate([x, y])

array([1, 2, 3, 4, 5, 6])

NUMPY 2D ROW-WISE (axis=0)

In [552]:
A = np.ones((2, 3))
B = np.zeros((2, 3))

np.concatenate([A, B], axis=0)

array([[1., 1., 1.],
       [1., 1., 1.],
       [0., 0., 0.],
       [0., 0., 0.]])

NUMPY 2D COLUMN-WISE (axis=1)

In [553]:
np.concatenate([A, B], axis=1)

array([[1., 1., 1., 0., 0., 0.],
       [1., 1., 1., 0., 0., 0.]])

pd.concat WITH SERIES

In [554]:
s1 = df_a["sepal length (cm)"]
s2 = df_b["sepal length (cm)"]

series_combined = pd.concat([s1, s2])
series_combined

,sepal length (cm)
0,5.1
1,4.9
2,4.7
3,4.6
4,5.0
...,...
95,5.7
96,5.7
97,6.2
98,5.1


pd.concat WITH DATAFRAMES (ROW-WISE)

In [555]:
df_concat = pd.concat([df_a, df_b, df_c], axis=0)
df_concat

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


COLUMN-WISE CONCAT (axis=1)

In [556]:
df_col = pd.concat(
    [df_a.reset_index(drop=True), df_b.reset_index(drop=True)],
    axis=1
)
df_col.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0,7.0,3.2,4.7,1.4,1
1,4.9,3.0,1.4,0.2,0,6.4,3.2,4.5,1.5,1
2,4.7,3.2,1.3,0.2,0,6.9,3.1,4.9,1.5,1
3,4.6,3.1,1.5,0.2,0,5.5,2.3,4.0,1.3,1
4,5.0,3.6,1.4,0.2,0,6.5,2.8,4.6,1.5,1


INDEX BEHAVIOR

In [557]:
df_duplicate_index = pd.concat([df_a, df_a])
df_duplicate_index

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
45,4.8,3.0,1.4,0.3,0
46,5.1,3.8,1.6,0.2,0
47,4.6,3.2,1.4,0.2,0
48,5.3,3.7,1.5,0.2,0


In [558]:
df_reset_index = pd.concat([df_a, df_b], ignore_index=True)
df_reset_index

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
95,5.7,3.0,4.2,1.2,1
96,5.7,2.9,4.2,1.3,1
97,6.2,2.9,4.3,1.3,1
98,5.1,2.5,3.0,1.1,1


COLUMN ALIGNMENT (JOIN LOGIC)

UNION COLUMNS

In [559]:
left = df_a.iloc[:, :3]
right = df_a.iloc[:, 2:5]

outer_join = pd.concat([left, right], axis=1, join="outer")
outer_join.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,1.4,0.2,0
1,4.9,3.0,1.4,1.4,0.2,0
2,4.7,3.2,1.3,1.3,0.2,0
3,4.6,3.1,1.5,1.5,0.2,0
4,5.0,3.6,1.4,1.4,0.2,0


INNER JOIN (INTERSECTION COLUMNS)

In [560]:
inner_join = pd.concat([left, right], axis=1, join="inner")
inner_join.head()


,sepal length (cm),sepal width (cm),petal length (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,1.4,0.2,0
1,4.9,3.0,1.4,1.4,0.2,0
2,4.7,3.2,1.3,1.3,0.2,0
3,4.6,3.1,1.5,1.5,0.2,0
4,5.0,3.6,1.4,1.4,0.2,0


 MULTIINDEX CONCAT USING KEYS

In [561]:
df_keys = pd.concat(
    [df_a, df_b],
    keys=["block_A", "block_B"]
)
df_keys

sepal length (cm)  sepal width (cm)  petal length (cm)  \
block_A 0                 5.1               3.5                1.4   
        1                 4.9               3.0                1.4   
        2                 4.7               3.2                1.3   
        3                 4.6               3.1                1.5   
        4                 5.0               3.6                1.4   
...                       ...               ...                ...   
block_B 95                5.7               3.0                4.2   
        96                5.7               2.9                4.2   
        97                6.2               2.9                4.3   
        98                5.1               2.5                3.0   
        99                5.7               2.8                4.1   

            petal width (cm)  target  
block_A 0                0.2       0  
        1                0.2       0  
        2                0.2       0  
        3                0.2       0  
        4                0.2       0  
...                      ...     ...  
block_B 95               1.2       1  
        96               1.3       1  
        97               1.3       1  
        98               1.1       1  
        99               1.3       1  

[100 rows x 5 columns]

APPEND (LEGACY BEHAVIOR REPLACED)

In [562]:
df_append_style = pd.concat([df_a, df_b], axis=0)
df_append_style

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
95,5.7,3.0,4.2,1.2,1
96,5.7,2.9,4.2,1.3,1
97,6.2,2.9,4.3,1.3,1
98,5.1,2.5,3.0,1.1,1


## Key Insight

- `pd.concat` is the core tool for dataset combination  
- `axis=0` stacks rows, `axis=1` stacks columns  
- `ignore_index` removes duplicate index structure  
- `keys` create hierarchical MultiIndex labels  
- `join='outer'` preserves all columns, `join='inner'` restricts overlap  
- `df.append()` is deprecated and fully replaced by `pd.concat()`  
- NumPy concat logic underpins Pandas behavior  

##3. Combining Datasets: Merge and Join

Pandas provides powerful in-memory merge and join operations for combining datasets efficiently.

Core function: `pd.merge()`

Similar to database joins (SQL-style operations)

Purpose:
- Combine related datasets using common keys or columns  
- Handle overlapping or related information across tables  

Key idea:
- Works like relational database joins  
- Matches rows based on shared identifiers (keys)  

Use cases:
- Linking datasets from different sources  
- Combining structured data for analysis  
- Resolving relationships between tables  

Advantage:
- High-performance and optimized for large datasets in memory  

##3.1 Relational Algebra

`pd.merge()` is based on relational algebra, a formal system for manipulating relational data.

Relational algebra:
- Provides a set of basic operations for working with structured data  
- Forms the theoretical foundation of most database systems  

Key idea:
- Complex data operations are built from a few simple primitives  
- These primitives can be efficiently combined for advanced queries  

In Pandas:
- `pd.merge()` and `.join()` implement core relational operations  
- Enable linking and combining datasets from different sources  

Importance:
- Allows structured, efficient, and scalable data integration  
- Bridges Pandas workflows with database-style logic  

 LOAD IRIS DATASET

In [563]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)
df["target"] = iris.target


add an explicit key to simulate relational structure

In [564]:
df["id"] = np.arange(len(df))
df["id"]

,id
0,0
1,1
2,2
3,3
4,4
...,...
145,145
146,146
147,147
148,148


CREATE TWO "RELATIONS" (TABLES)

Left Table

In [565]:
left = df[["id", "sepal length (cm)", "sepal width (cm)", "target"]].iloc[:100]
left

,id,sepal length (cm),sepal width (cm),target
0,0,5.1,3.5,0
1,1,4.9,3.0,0
2,2,4.7,3.2,0
3,3,4.6,3.1,0
4,4,5.0,3.6,0
...,...,...,...,...
95,95,5.7,3.0,1
96,96,5.7,2.9,1
97,97,6.2,2.9,1
98,98,5.1,2.5,1


Right Table

In [566]:
right = df[["id", "petal length (cm)", "petal width (cm)"]].iloc[50:150]
right

,id,petal length (cm),petal width (cm)
50,50,4.7,1.4
51,51,4.5,1.5
52,52,4.9,1.5
53,53,4.0,1.3
54,54,4.6,1.5
...,...,...,...
145,145,5.2,2.3
146,146,5.0,1.9
147,147,5.2,2.0
148,148,5.4,2.3


INNER JOIN (RELATIONAL ALGEBRA CORE OPERATION)

In [567]:
inner_join = pd.merge(left, right, on="id", how="inner")
inner_join

,id,sepal length (cm),sepal width (cm),target,petal length (cm),petal width (cm)
0,50,7.0,3.2,1,4.7,1.4
1,51,6.4,3.2,1,4.5,1.5
2,52,6.9,3.1,1,4.9,1.5
3,53,5.5,2.3,1,4.0,1.3
4,54,6.5,2.8,1,4.6,1.5
5,55,5.7,2.8,1,4.5,1.3
6,56,6.3,3.3,1,4.7,1.6
7,57,4.9,2.4,1,3.3,1.0
8,58,6.6,2.9,1,4.6,1.3
9,59,5.2,2.7,1,3.9,1.4


LEFT JOIN

In [568]:
left_join = pd.merge(left, right, on="id", how="left")
left_join

,id,sepal length (cm),sepal width (cm),target,petal length (cm),petal width (cm)
0,0,5.1,3.5,0,NaN,NaN
1,1,4.9,3.0,0,NaN,NaN
2,2,4.7,3.2,0,NaN,NaN
3,3,4.6,3.1,0,NaN,NaN
4,4,5.0,3.6,0,NaN,NaN
...,...,...,...,...,...,...
95,95,5.7,3.0,1,4.2,1.2
96,96,5.7,2.9,1,4.2,1.3
97,97,6.2,2.9,1,4.3,1.3
98,98,5.1,2.5,1,3.0,1.1


OUTER JOIN

In [569]:
outer_join = pd.merge(left, right, on="id", how="outer")
outer_join

,id,sepal length (cm),sepal width (cm),target,petal length (cm),petal width (cm)
0,0,5.1,3.5,0.0,NaN,NaN
1,1,4.9,3.0,0.0,NaN,NaN
2,2,4.7,3.2,0.0,NaN,NaN
3,3,4.6,3.1,0.0,NaN,NaN
4,4,5.0,3.6,0.0,NaN,NaN
...,...,...,...,...,...,...
145,145,NaN,NaN,NaN,5.2,2.3
146,146,NaN,NaN,NaN,5.0,1.9
147,147,NaN,NaN,NaN,5.2,2.0
148,148,NaN,NaN,NaN,5.4,2.3


## Key Insight

- Data tables are treated as relations (sets of tuples)  
- Core operations: SELECT, PROJECT, JOIN  
- `pd.merge()` implements JOIN logic in Pandas  
- Matching is performed on keys (here: 'id')  
- INNER JOIN keeps only matching records  
- OUTER JOIN preserves all records with NaN for missing matches  
- Complex queries are compositions of simple relational primitives  

##3.2 Categories of Joins

`pd.merge()` supports three main join types depending on the structure of the data:

---

### 3.2.1 One-to-one joins
Each key appears only once in both datasets.

- Works like simple column-wise combination  
- Produces a single merged row per matching entry  
- Example idea: employee details matched with hire dates  

---

### 3.2.2 Many-to-one joins
One dataset has duplicate keys, the other does not.

- Repeated values are carried into the result  
- Example idea: multiple employees linked to one supervisor per group  

---

### 3.2.3 Many-to-many joins
Both datasets contain duplicate keys.

- Every possible combination of matches is created  
- Can significantly expand the dataset size  
- Example idea: employees linked to multiple skills per group  

---

### Key idea

- Same `pd.merge()` function handles all cases  
- Join type is determined automatically by data structure  
- Real-world data often involves many-to-one or many-to-many relationships rather than clean one-to-one matches  

LOAD IRIS DATASET

In [570]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)
df["target"] = iris.target

# add a synthetic key to control relational structure

In [571]:
df["id"] = np.arange(len(df))

ONE-TO-ONE JOIN

In [572]:
# each id appears once in both tables
left_1to1 = df[["id", "sepal length (cm)", "target"]].iloc[:50]
right_1to1 = df[["id", "sepal width (cm)"]].iloc[:50]

one_to_one = pd.merge(left_1to1, right_1to1, on="id", how="inner")
one_to_one

,id,sepal length (cm),target,sepal width (cm)
0,0,5.1,0,3.5
1,1,4.9,0,3.0
2,2,4.7,0,3.2
3,3,4.6,0,3.1
4,4,5.0,0,3.6
5,5,5.4,0,3.9
6,6,4.6,0,3.4
7,7,5.0,0,3.4
8,8,4.4,0,2.9
9,9,4.9,0,3.1


MANY-TO-ONE JOIN

In [573]:
# right table has repeated keys (group-level feature)
left_m2o = df[["id", "sepal length (cm)"]].iloc[:50]

right_m2o = df[["target"]].iloc[:3].copy()
right_m2o["id"] = [0, 1, 2]  # small key set reused conceptually

# artificially repeat right-side keys to simulate many-to-one structure
right_m2o = pd.concat([right_m2o] * 20, ignore_index=True)
right_m2o["id"] = np.resize([0, 1, 2], len(right_m2o))

many_to_one = pd.merge(left_m2o, right_m2o, on="id", how="inner")
many_to_one

,id,sepal length (cm),target
0,0,5.1,0
1,0,5.1,0
2,0,5.1,0
3,0,5.1,0
4,0,5.1,0
5,0,5.1,0
6,0,5.1,0
7,0,5.1,0
8,0,5.1,0
9,0,5.1,0


MANY-TO-MANY JOIN

In [574]:
# both sides have duplicate keys
left_m2m = df[["id", "sepal length (cm)"]].iloc[:10].copy()
left_m2m["group"] = [0, 0, 1, 1, 2, 2, 3, 3, 4, 4]

right_m2m = df[["id", "petal length (cm)"]].iloc[:10].copy()
right_m2m["group"] = [0, 0, 1, 1, 2, 2, 3, 3, 4, 4]

many_to_many = pd.merge(left_m2m, right_m2m, on="group", how="inner")
many_to_many

,id_x,sepal length (cm),group,id_y,petal length (cm)
0,0,5.1,0,0,1.4
1,0,5.1,0,1,1.4
2,1,4.9,0,0,1.4
3,1,4.9,0,1,1.4
4,2,4.7,1,2,1.3
5,2,4.7,1,3,1.5
6,3,4.6,1,2,1.3
7,3,4.6,1,3,1.5
8,4,5.0,2,4,1.4
9,4,5.0,2,5,1.7


## Key Insight

- One-to-one: unique keys on both sides → 1 match per record  
- Many-to-one: repeated keys on one side → values are broadcast  
- Many-to-many: repeated keys on both sides → Cartesian-like expansion  
- `pd.merge()` automatically adapts based on key structure  
- Real datasets rarely satisfy strict one-to-one assumptions  

##3.3 Specification of the Merge Key

`pd.merge()` allows flexible ways to define how datasets are matched.

---

###3.3.1 Default behavior
- Automatically matches columns with the same name  
- Uses shared column names as merge keys  

---

### 3.3.2 `on` keyword
- Explicitly specifies the column(s) to join on  
- Works only when both datasets share the same column name  

---

### 3.3.3 `left_on` and `right_on`
- Used when column names differ between datasets  
- Maps different column names to act as join keys  
- May create extra columns that can be removed afterward if needed  

---

### 3.3.4 `left_index` and `right_index`
- Uses index instead of columns as merge keys  
- Useful when identifiers are stored in DataFrame indices  

---

### 3.3.5 Mixing index and column joins
- Can combine:
  - index on one side  
  - column on the other side  
- Enables flexible alignment across differently structured datasets  

---

### Key idea
Merge behavior is fully controlled by how keys are specified, making Pandas flexible for real-world messy data structures.

LOAD IRIS DATASET

In [575]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)
df["target"] = iris.target


create explicit ID column for key-based operations

In [576]:
df["id"] = np.arange(len(df))

split into two relational tables

In [577]:
left = df.iloc[:100].copy()
right = df.iloc[50:150].copy()

DEFAULT BEHAVIOR (AUTO MATCHING COLUMNS)

In [578]:
# here both share "id", so it is inferred automatically
default_merge = pd.merge(left, right)
default_merge

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,id
0,7.0,3.2,4.7,1.4,1,50
1,6.4,3.2,4.5,1.5,1,51
2,6.9,3.1,4.9,1.5,1,52
3,5.5,2.3,4.0,1.3,1,53
4,6.5,2.8,4.6,1.5,1,54
5,5.7,2.8,4.5,1.3,1,55
6,6.3,3.3,4.7,1.6,1,56
7,4.9,2.4,3.3,1.0,1,57
8,6.6,2.9,4.6,1.3,1,58
9,5.2,2.7,3.9,1.4,1,59


 EXPLICIT 'on' KEY

In [579]:
on_merge = pd.merge(left, right, on="id")
on_merge

,sepal length (cm)_x,sepal width (cm)_x,petal length (cm)_x,petal width (cm)_x,target_x,id,sepal length (cm)_y,sepal width (cm)_y,petal length (cm)_y,petal width (cm)_y,target_y
0,7.0,3.2,4.7,1.4,1,50,7.0,3.2,4.7,1.4,1
1,6.4,3.2,4.5,1.5,1,51,6.4,3.2,4.5,1.5,1
2,6.9,3.1,4.9,1.5,1,52,6.9,3.1,4.9,1.5,1
3,5.5,2.3,4.0,1.3,1,53,5.5,2.3,4.0,1.3,1
4,6.5,2.8,4.6,1.5,1,54,6.5,2.8,4.6,1.5,1
5,5.7,2.8,4.5,1.3,1,55,5.7,2.8,4.5,1.3,1
6,6.3,3.3,4.7,1.6,1,56,6.3,3.3,4.7,1.6,1
7,4.9,2.4,3.3,1.0,1,57,4.9,2.4,3.3,1.0,1
8,6.6,2.9,4.6,1.3,1,58,6.6,2.9,4.6,1.3,1
9,5.2,2.7,3.9,1.4,1,59,5.2,2.7,3.9,1.4,1


left_on and right_on (DIFFERENT COLUMN NAMES)

In [580]:
left_renamed = left.rename(columns={"id": "left_id"})
right_renamed = right.rename(columns={"id": "right_id"})

lr_merge = pd.merge(
    left_renamed,
    right_renamed,
    left_on="left_id",
    right_on="right_id"
)

print("\n--left_on / right_on MERGE--")
lr_merge.head()


--left_on / right_on MERGE--


,sepal length (cm)_x,sepal width (cm)_x,petal length (cm)_x,petal width (cm)_x,target_x,left_id,sepal length (cm)_y,sepal width (cm)_y,petal length (cm)_y,petal width (cm)_y,target_y,right_id
0,7.0,3.2,4.7,1.4,1,50,7.0,3.2,4.7,1.4,1,50
1,6.4,3.2,4.5,1.5,1,51,6.4,3.2,4.5,1.5,1,51
2,6.9,3.1,4.9,1.5,1,52,6.9,3.1,4.9,1.5,1,52
3,5.5,2.3,4.0,1.3,1,53,5.5,2.3,4.0,1.3,1,53
4,6.5,2.8,4.6,1.5,1,54,6.5,2.8,4.6,1.5,1,54


MERGE USING INDEX (left_index / right_index)

In [581]:
left_indexed = left.set_index("id")
right_indexed = right.set_index("id")

index_merge = pd.merge(
    left_indexed,
    right_indexed,
    left_index=True,
    right_index=True,
    suffixes=("_left", "_right")
)

index_merge

,sepal length (cm)_left,sepal width (cm)_left,petal length (cm)_left,petal width (cm)_left,target_left,sepal length (cm)_right,sepal width (cm)_right,petal length (cm)_right,petal width (cm)_right,target_right
id,,,,,,,,,,
50,7.0,3.2,4.7,1.4,1,7.0,3.2,4.7,1.4,1
51,6.4,3.2,4.5,1.5,1,6.4,3.2,4.5,1.5,1
52,6.9,3.1,4.9,1.5,1,6.9,3.1,4.9,1.5,1
53,5.5,2.3,4.0,1.3,1,5.5,2.3,4.0,1.3,1
54,6.5,2.8,4.6,1.5,1,6.5,2.8,4.6,1.5,1
55,5.7,2.8,4.5,1.3,1,5.7,2.8,4.5,1.3,1
56,6.3,3.3,4.7,1.6,1,6.3,3.3,4.7,1.6,1
57,4.9,2.4,3.3,1.0,1,4.9,2.4,3.3,1.0,1
58,6.6,2.9,4.6,1.3,1,6.6,2.9,4.6,1.3,1


MIXING INDEX AND COLUMN

In [582]:
mixed_merge = pd.merge(
    left_indexed,
    right,
    left_index=True,
    right_on="id"
)
mixed_merge

,sepal length (cm)_x,sepal width (cm)_x,petal length (cm)_x,petal width (cm)_x,target_x,sepal length (cm)_y,sepal width (cm)_y,petal length (cm)_y,petal width (cm)_y,target_y,id
50,7.0,3.2,4.7,1.4,1,7.0,3.2,4.7,1.4,1,50
51,6.4,3.2,4.5,1.5,1,6.4,3.2,4.5,1.5,1,51
52,6.9,3.1,4.9,1.5,1,6.9,3.1,4.9,1.5,1,52
53,5.5,2.3,4.0,1.3,1,5.5,2.3,4.0,1.3,1,53
54,6.5,2.8,4.6,1.5,1,6.5,2.8,4.6,1.5,1,54
55,5.7,2.8,4.5,1.3,1,5.7,2.8,4.5,1.3,1,55
56,6.3,3.3,4.7,1.6,1,6.3,3.3,4.7,1.6,1,56
57,4.9,2.4,3.3,1.0,1,4.9,2.4,3.3,1.0,1,57
58,6.6,2.9,4.6,1.3,1,6.6,2.9,4.6,1.3,1,58
59,5.2,2.7,3.9,1.4,1,5.2,2.7,3.9,1.4,1,59


## Key Insight

- merge keys define how datasets align structurally  
- default: shared column names are auto-detected  
- 'on' explicitly sets shared key column(s)  
- left_on/right_on handle mismatched column names  
- left_index/right_index enable index-based joins  
- mixing index and column keys allows flexible real-world integration  

##3.4 Specifying Set Arithmetic for Joins

When merging datasets, the `how` parameter controls how unmatched keys are handled.

---

### 3.4.1 Inner join (default)
- Keeps only common keys in both datasets  
- Intersection of sets  
- Drops non-matching rows  

---

### 3.4.2 Outer join
- Keeps all keys from both datasets  
- Union of sets  
- Missing values are filled with NaN  

---

### 3.4.3 Left join
- Keeps all keys from the left dataset  
- Matches from right are added where available  
- Missing matches become NaN  

---

### 3.4.4 Right join
- Keeps all keys from the right dataset  
- Matches from left are added where available  
- Missing matches become NaN  

---

## Key idea

Set arithmetic determines whether you prioritize:

- shared data (inner)  
- completeness (outer)  
- one dataset as the reference frame (left/right)  

LOAD IRIS DATASET

In [583]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)
df["target"] = iris.target

# create explicit key for controlled joins
df["id"] = np.arange(len(df))

In [584]:
# split into two overlapping relational tables
left = df.iloc[:100].copy()
right = df.iloc[50:150].copy()

INNER JOIN (INTERSECTION)

In [585]:
inner = pd.merge(left, right, on="id", how="inner")
inner

,sepal length (cm)_x,sepal width (cm)_x,petal length (cm)_x,petal width (cm)_x,target_x,id,sepal length (cm)_y,sepal width (cm)_y,petal length (cm)_y,petal width (cm)_y,target_y
0,7.0,3.2,4.7,1.4,1,50,7.0,3.2,4.7,1.4,1
1,6.4,3.2,4.5,1.5,1,51,6.4,3.2,4.5,1.5,1
2,6.9,3.1,4.9,1.5,1,52,6.9,3.1,4.9,1.5,1
3,5.5,2.3,4.0,1.3,1,53,5.5,2.3,4.0,1.3,1
4,6.5,2.8,4.6,1.5,1,54,6.5,2.8,4.6,1.5,1
5,5.7,2.8,4.5,1.3,1,55,5.7,2.8,4.5,1.3,1
6,6.3,3.3,4.7,1.6,1,56,6.3,3.3,4.7,1.6,1
7,4.9,2.4,3.3,1.0,1,57,4.9,2.4,3.3,1.0,1
8,6.6,2.9,4.6,1.3,1,58,6.6,2.9,4.6,1.3,1
9,5.2,2.7,3.9,1.4,1,59,5.2,2.7,3.9,1.4,1


OUTER JOIN (UNION)

In [586]:
outer = pd.merge(left, right, on="id", how="outer")
outer

,sepal length (cm)_x,sepal width (cm)_x,petal length (cm)_x,petal width (cm)_x,target_x,id,sepal length (cm)_y,sepal width (cm)_y,petal length (cm)_y,petal width (cm)_y,target_y
0,5.1,3.5,1.4,0.2,0.0,0,NaN,NaN,NaN,NaN,NaN
1,4.9,3.0,1.4,0.2,0.0,1,NaN,NaN,NaN,NaN,NaN
2,4.7,3.2,1.3,0.2,0.0,2,NaN,NaN,NaN,NaN,NaN
3,4.6,3.1,1.5,0.2,0.0,3,NaN,NaN,NaN,NaN,NaN
4,5.0,3.6,1.4,0.2,0.0,4,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
145,NaN,NaN,NaN,NaN,NaN,145,6.7,3.0,5.2,2.3,2.0
146,NaN,NaN,NaN,NaN,NaN,146,6.3,2.5,5.0,1.9,2.0
147,NaN,NaN,NaN,NaN,NaN,147,6.5,3.0,5.2,2.0,2.0
148,NaN,NaN,NaN,NaN,NaN,148,6.2,3.4,5.4,2.3,2.0


LEFT JOIN (LEFT AS REFERENCE)

In [587]:
left_join = pd.merge(left, right, on="id", how="left")
left_join

,sepal length (cm)_x,sepal width (cm)_x,petal length (cm)_x,petal width (cm)_x,target_x,id,sepal length (cm)_y,sepal width (cm)_y,petal length (cm)_y,petal width (cm)_y,target_y
0,5.1,3.5,1.4,0.2,0,0,NaN,NaN,NaN,NaN,NaN
1,4.9,3.0,1.4,0.2,0,1,NaN,NaN,NaN,NaN,NaN
2,4.7,3.2,1.3,0.2,0,2,NaN,NaN,NaN,NaN,NaN
3,4.6,3.1,1.5,0.2,0,3,NaN,NaN,NaN,NaN,NaN
4,5.0,3.6,1.4,0.2,0,4,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
95,5.7,3.0,4.2,1.2,1,95,5.7,3.0,4.2,1.2,1.0
96,5.7,2.9,4.2,1.3,1,96,5.7,2.9,4.2,1.3,1.0
97,6.2,2.9,4.3,1.3,1,97,6.2,2.9,4.3,1.3,1.0
98,5.1,2.5,3.0,1.1,1,98,5.1,2.5,3.0,1.1,1.0


RIGHT JOIN (RIGHT AS REFERENCE)

In [588]:
right_join = pd.merge(left, right, on="id", how="right")
right_join

,sepal length (cm)_x,sepal width (cm)_x,petal length (cm)_x,petal width (cm)_x,target_x,id,sepal length (cm)_y,sepal width (cm)_y,petal length (cm)_y,petal width (cm)_y,target_y
0,7.0,3.2,4.7,1.4,1.0,50,7.0,3.2,4.7,1.4,1
1,6.4,3.2,4.5,1.5,1.0,51,6.4,3.2,4.5,1.5,1
2,6.9,3.1,4.9,1.5,1.0,52,6.9,3.1,4.9,1.5,1
3,5.5,2.3,4.0,1.3,1.0,53,5.5,2.3,4.0,1.3,1
4,6.5,2.8,4.6,1.5,1.0,54,6.5,2.8,4.6,1.5,1
...,...,...,...,...,...,...,...,...,...,...,...
95,NaN,NaN,NaN,NaN,NaN,145,6.7,3.0,5.2,2.3,2
96,NaN,NaN,NaN,NaN,NaN,146,6.3,2.5,5.0,1.9,2
97,NaN,NaN,NaN,NaN,NaN,147,6.5,3.0,5.2,2.0,2
98,NaN,NaN,NaN,NaN,NaN,148,6.2,3.4,5.4,2.3,2


## Set Arithmetic Summary

- INNER JOIN  → intersection of keys (A ∩ B)  
- OUTER JOIN  → union of keys (A ∪ B)  
- LEFT JOIN   → preserve left set (A + matches in B)  
- RIGHT JOIN  → preserve right set (B + matches in A)  
- Missing matches are filled with NaN in non-inner joins  
- Choice of 'how' defines analytical perspective, not just structure  

##3.5 Overlapping Column Names: `suffixes` Keyword (short notes)

When two DataFrames share column names (besides the join key), Pandas must avoid ambiguity in the result.

---

###3.5.1 Default behavior
Pandas automatically renames conflicting columns:

- `_x` → from the left DataFrame  
- `_y` → from the right DataFrame  

This ensures column names remain unique after merging.

---

###3.5.2 Custom control
You can explicitly define your own suffixes using the `suffixes` option.

This is useful when `_x` and `_y` are not meaningful in your context.

---

## Key idea

Suffixes are simply labels for distinguishing overlapping attributes after a join. They do not change the data only how it is interpreted and displayed.

LOAD IRIS DATASET

In [589]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)
df["target"] = iris.target
df["id"] = np.arange(len(df))

CREATE OVERLAPPING DATASETS (COMMON COLUMN NAMES)

In [590]:
left = df.iloc[:100].copy()
right = df.iloc[50:150].copy()

# introduce overlapping feature names explicitly
left_small = left[["id", "sepal length (cm)", "sepal width (cm)", "target"]]
right_small = right[["id", "sepal length (cm)", "petal length (cm)", "target"]]

DEFAULT SUFFIX BEHAVIOR (_x, _y)

In [591]:
default_merge = pd.merge(left_small, right_small, on="id", how="inner")
default_merge

,id,sepal length (cm)_x,sepal width (cm),target_x,sepal length (cm)_y,petal length (cm),target_y
0,50,7.0,3.2,1,7.0,4.7,1
1,51,6.4,3.2,1,6.4,4.5,1
2,52,6.9,3.1,1,6.9,4.9,1
3,53,5.5,2.3,1,5.5,4.0,1
4,54,6.5,2.8,1,6.5,4.6,1
5,55,5.7,2.8,1,5.7,4.5,1
6,56,6.3,3.3,1,6.3,4.7,1
7,57,4.9,2.4,1,4.9,3.3,1
8,58,6.6,2.9,1,6.6,4.6,1
9,59,5.2,2.7,1,5.2,3.9,1


CUSTOM SUFFIX CONTROL

In [592]:
custom_merge = pd.merge(
    left_small,
    right_small,
    on="id",
    how="inner",
    suffixes=("_left", "_right")
)
custom_merge

,id,sepal length (cm)_left,sepal width (cm),target_left,sepal length (cm)_right,petal length (cm),target_right
0,50,7.0,3.2,1,7.0,4.7,1
1,51,6.4,3.2,1,6.4,4.5,1
2,52,6.9,3.1,1,6.9,4.9,1
3,53,5.5,2.3,1,5.5,4.0,1
4,54,6.5,2.8,1,6.5,4.6,1
5,55,5.7,2.8,1,5.7,4.5,1
6,56,6.3,3.3,1,6.3,4.7,1
7,57,4.9,2.4,1,4.9,3.3,1
8,58,6.6,2.9,1,6.6,4.6,1
9,59,5.2,2.7,1,5.2,3.9,1


## Key Insight

- Overlapping column names create ambiguity after merge  
- Pandas resolves this using suffixes  
- Default: _x (left), _y (right)  
- Custom suffixes improve interpretability  
- Suffixing does NOT alter data, only labeling structure  